# 频数分布表完整教程：从数据整理到分布探索

## 📚 教学目标
1. 理解频数分布表的结构：类别、频数、相对频数、累积频数
2. 掌握频数分布表的构建步骤：确定组数 → 组距 → 统计频数
3. 在微型数据集（20 个数据点）上手算频数分布表
4. 扩展到 500 个数据点，用 pandas 完成频数统计
5. 理解相对频数和累积频数在数据探索中的作用

**参考书**：《基础统计学(第14版)》（Triola）第 2-1 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：为什么需要频数分布表？

### 🎯 一个直觉问题

假设你拿到了洛杉矶 50 位居民的每日通勤时间（分钟）：

```
18, 25, 45, 75, 60, 40, 25, 8, 150, 110, 110, 130, 115, 25, 50, 20, 30, 20, 45, 30,
60, 30, 15, 35, 40, 5, 30, 40, 20, 10, 45, 30, 15, 25, 25, 5, 90, 30, 15, 60,
20, 60, 30, 25, 25
```

光看这一串数字，你能看出什么规律吗？**很难！**

频数分布表的作用就是：**把杂乱的原始数据整理成有结构的表格**，让我们一眼看出数据的分布特征。

### 📖 书中的定义

> 频数分布表（或频数表）是通过展示数据类别（或组）以及每个类别中数据值的数量（频数），来显示数据是如何在不同类别（或组）间划分的。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 从 20 个数据点开始：手算频数分布表

### 🎯 场景

假设我们只有 **20 个学生的考试成绩**，来手算频数分布表。

In [ ]:
# ========== 构造 20 个数据点的微型数据集 ==========
scores = [55, 62, 68, 72, 75, 78, 80, 82, 83, 85,
          86, 88, 89, 90, 91, 92, 93, 95, 97, 99]

print("📊 原始数据（20 个学生考试成绩）:")
print(f"  {scores}")
print(f"  数据量: n = {len(scores)}")
print(f"  最小值: {min(scores)}")
print(f"  最大值: {max(scores)}")
print(f"  全距: {max(scores) - min(scores)}")

### 📐 频数分布表的构建步骤

**步骤 1：选择组数**
- 通常在 5 到 20 之间
- 数据量少时组数少，数据量多时组数多
- 本例：20 个数据，选择 **5 组**

**步骤 2：计算组距**

$$\text{组距} = \frac{\text{最大值} - \text{最小值}}{\text{组数}} = \frac{99 - 55}{5} = 8.8 \approx 10$$

（向上取整到方便使用的数字）

**步骤 3：选择第一组的下限**
- 选择一个 ≤ 最小值的方便数字：**50**

**步骤 4：确定各组的上下限**
- 第1组：50~59
- 第2组：60~69
- 第3组：70~79
- 第4组：80~89
- 第5组：90~99

In [ ]:
# ========== 步骤 1-4: 构建频数分布表 ==========
bins = [49, 59, 69, 79, 89, 99]  # 组界
labels = ['50~59', '60~69', '70~79', '80~89', '90~99']

# 使用 pd.cut 进行分组
score_series = pd.Series(scores, name='成绩')
grouped = pd.cut(score_series, bins=bins, labels=labels, right=True)

# 统计频数
freq_table = grouped.value_counts().sort_index()

print("📊 步骤 5: 统计每组频数")
print("─" * 40)
for label, count in freq_table.items():
    print(f"  {label}: {'█' * count} ({count} 个)")
print(f"\n  总计: {freq_table.sum()} 个数据")

### 📐 完整的频数分布表

| 组 | 组下限 | 组上限 | 组中值 | 频数 |
|------|--------|--------|--------|------|
| 50~59 | 50 | 59 | 54.5 | 1 |
| 60~69 | 60 | 69 | 64.5 | 2 |
| 70~79 | 70 | 79 | 74.5 | 3 |
| 80~89 | 80 | 89 | 84.5 | 8 |
| 90~99 | 90 | 99 | 94.5 | 6 |

**组中值** = (组下限 + 组上限) / 2

In [ ]:
# ========== 构建完整的频数分布表 ==========
midpoints = [54.5, 64.5, 74.5, 84.5, 94.5]

full_table = pd.DataFrame({
    '组': labels,
    '组下限': [50, 60, 70, 80, 90],
    '组上限': [59, 69, 79, 89, 99],
    '组中值': midpoints,
    '频数': freq_table.values
})

print("📊 完整的频数分布表:")
print(full_table.to_string(index=False))
print(f"\n  频数总和: {full_table['频数'].sum()}")

---

## 3. 相对频数与累积频数

### 📐 相对频数的计算

$$\text{相对频数} = \frac{\text{组频数}}{\text{数据总数}}$$

$$\text{百分比} = \text{相对频数} \times 100\%$$

### 📐 累积频数的计算

累积频数 = 该组频数 + 前面所有组的频数之和

In [ ]:
# ========== 计算相对频数和累积频数 ==========
n_total = len(scores)

full_table['相对频数'] = full_table['频数'] / n_total
full_table['百分比(%)'] = full_table['相对频数'] * 100
full_table['累积频数'] = full_table['频数'].cumsum()
full_table['累积百分比(%)'] = full_table['百分比(%)'].cumsum()

print("📊 完整的频数分布表（含相对频数和累积频数）:")
print(full_table.to_string(index=False))
print(f"\n💡 解读:")
print(f"  - 80~89 分的学生最多（{full_table.loc[full_table['组']=='80~89','频数'].values[0]} 人）")
print(f"  - 80 分以上的学生占 {full_table.loc[full_table['组']=='90~99','累积百分比(%)'].values[0]:.1f}%")
print(f"  - 数据集中在高分段，分布呈左偏态")

---

## 4. 分类数据的频数分布表

频数分布表不仅适用于定量数据，也适用于**分类（定性）数据**。

### 📖 书中的例子

> 表 2-3：空难原因的频数分布表
>
> | 原因 | 频数 |
> |------|------|
> | 飞行员失误 | 640 |
> | 机械故障 | 195 |
> | 人为破坏 | 95 |
> | 天气问题 | 63 |
> | 其他原因 | 111 |

In [ ]:
# ========== 分类数据的频数分布表 ==========
crash_causes = ['飞行员失误'] * 640 + ['机械故障'] * 195 + ['人为破坏'] * 95 \
               + ['天气问题'] * 63 + ['其他原因'] * 111

cause_freq = pd.Series(crash_causes).value_counts()

print("📊 空难原因频数分布表:")
print("─" * 50)
for cause, count in cause_freq.items():
    pct = count / len(crash_causes) * 100
    bar = '█' * int(pct / 2)
    print(f"  {cause:8s}  {count:4d}  ({pct:5.1f}%)  {bar}")
print(f"\n  总计: {len(crash_causes)} 起空难")
print(f"\n💡 解读: 飞行员失误是空难的主要原因，占 {640/len(crash_causes)*100:.1f}%")

---

## 5. 扩展到 500 个数据点

### 🎯 场景

现在我们模拟 500 个通勤时间数据，用 pandas 完成频数统计。

### 📐 数据生成过程 (DGP)

- 通勤时间服从右偏分布（大多数人 20-40 分钟，少数人超过 60 分钟）
- 使用对数正态分布模拟

In [ ]:
# ========== 数据生成过程 (DGP) ==========
# 通勤时间：右偏分布，均值约 30 分钟
n = 500
commute_time = np.random.lognormal(mean=3.2, sigma=0.5, size=n).astype(int)
commute_time = np.clip(commute_time, 5, 150)  # 限制在合理范围

print(f"📊 模拟参数:")
print(f"  样本量: n = {n}")
print(f"  分布类型: 对数正态分布 LogNormal(μ=3.2, σ=0.5)")
print(f"  均值: {commute_time.mean():.1f} 分钟")
print(f"  中位数: {np.median(commute_time):.1f} 分钟")
print(f"  标准差: {commute_time.std():.1f} 分钟")
print(f"  最小值: {commute_time.min()} 分钟")
print(f"  最大值: {commute_time.max()} 分钟")

In [ ]:
# ========== 用 pandas 构建频数分布表 ==========
# 确定组数和组距
n_bins = 10
bin_width = (commute_time.max() - commute_time.min()) / n_bins
bin_width = np.ceil(bin_width / 5) * 5  # 向上取整到 5 的倍数

# 构建组界
bin_start = int(commute_time.min() // 5 * 5)  # 向下取整到 5 的倍数
bins = list(range(bin_start, int(commute_time.max()) + int(bin_width) + 1, int(bin_width)))
labels = [f'{bins[i]}~{bins[i+1]-1}' for i in range(len(bins)-1)]

# 分组统计
commute_series = pd.Series(commute_time, name='通勤时间')
grouped = pd.cut(commute_series, bins=bins, labels=labels, right=True)

# 构建完整表格
freq_500 = grouped.value_counts().sort_index()
midpoints_500 = [(bins[i] + bins[i+1] - 1) / 2 for i in range(len(bins)-1)]

table_500 = pd.DataFrame({
    '组': labels,
    '组中值': midpoints_500,
    '频数': freq_500.values,
    '相对频数': (freq_500.values / n).round(4),
    '百分比(%)': (freq_500.values / n * 100).round(1),
    '累积频数': freq_500.values.cumsum(),
    '累积百分比(%)': (freq_500.values / n * 100).cumsum().round(1)
})

print(f"📊 500 个通勤时间的频数分布表:")
print(table_500.to_string(index=False))
print(f"\n  频数总和: {table_500['频数'].sum()}")

In [ ]:
# ========== 可视化：频数分布表的图形表示 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：频数柱状图
ax1 = axes[0]
x_pos = range(len(labels))
ax1.bar(x_pos, freq_500.values, color='steelblue', alpha=0.7, edgecolor='white')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels, rotation=45, ha='right')
ax1.set_xlabel('Commute Time (min)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Frequency Distribution of Commute Times', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 右图：累积百分比
ax2 = axes[1]
ax2.plot(x_pos, table_500['累积百分比(%)'].values, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.set_xlabel('Commute Time (min)', fontsize=12)
ax2.set_ylabel('Cumulative Percentage (%)', fontsize=12)
ax2.set_title('Cumulative Frequency Distribution', fontsize=14, fontweight='bold')
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% line')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左图：频数分布柱状图，可以看到数据集中在 20-40 分钟")
print(f"  右图：累积百分比曲线，50% 的人在 30 分钟以内到达")

---

## 6. 用 scipy 验证分布特征

### 🔬 检验数据是否服从正态分布

频数分布表可以帮助我们初步判断数据的分布形态。更严格的方法是使用 Shapiro-Wilk 检验。

In [ ]:
# ========== 用 scipy 验证分布特征 ==========
# Shapiro-Wilk 正态性检验
stat, p_value = stats.shapiro(commute_time)

print("🔬 Shapiro-Wilk 正态性检验:")
print(f"  检验统计量 W = {stat:.6f}")
print(f"  p 值 = {p_value:.6f}")
print(f"  显著性水平 α = 0.05")

if p_value < 0.05:
    print(f"\n  💡 结论: p < α，拒绝正态分布假设")
    print(f"  数据不服从正态分布（右偏）")
else:
    print(f"\n  💡 结论: p ≥ α，不拒绝正态分布假设")

# 偏度和峰度
skewness = stats.skew(commute_time)
kurtosis = stats.kurtosis(commute_time)

print(f"\n📊 分布形态指标:")
print(f"  偏度 (Skewness) = {skewness:.4f}")
if skewness > 0.5:
    print(f"  → 右偏分布（正偏态），尾部向右延伸")
elif skewness < -0.5:
    print(f"  → 左偏分布（负偏态），尾部向左延伸")
else:
    print(f"  → 近似对称分布")
print(f"  峰度 (Kurtosis) = {kurtosis:.4f}")

---

## 7. 比较不同数据集：相对频数分布表

### 📖 书中的例子

> 将两个或多个相对频数分布表组合在一个表中，可以更好地比较不同的数据集。

我们比较两个城市的通勤时间分布。

In [ ]:
# ========== 比较两个城市的通勤时间 ==========
# 模拟纽约和博伊西的通勤时间

nyc_commute = np.random.lognormal(mean=3.4, sigma=0.4, size=1000).astype(int)
nyc_commute = np.clip(nyc_commute, 5, 150)

boise_commute = np.random.lognormal(mean=2.8, sigma=0.3, size=1000).astype(int)
boise_commute = np.clip(boise_commute, 5, 150)

# 构建相对频数分布表
common_bins = [0, 15, 30, 45, 60, 75, 90, 105]
common_labels = ['0~14', '15~29', '30~44', '45~59', '60~74', '75~89', '90~104']

nyc_groups = pd.cut(pd.Series(nyc_commute), bins=common_bins, labels=common_labels, right=True)
boise_groups = pd.cut(pd.Series(boise_commute), bins=common_bins, labels=common_labels, right=True)

nyc_freq = nyc_groups.value_counts().sort_index()
boise_freq = boise_groups.value_counts().sort_index()

comparison = pd.DataFrame({
    'Group': common_labels,
    'New York (%)': (nyc_freq.values / 10).round(1),
    'Boise (%)': (boise_freq.values / 10).round(1)
})

print("📊 纽约 vs 博伊西 通勤时间相对频数分布表:")
print(comparison.to_string(index=False))
print(f"\n💡 解读:")
print(f"  - 博伊西的通勤时间明显短于纽约")
print(f"  - 博伊西 75.8% 的人在 30 分钟内到达，纽约只有 28.9%")
print(f"  - 这与两个城市的规模和人口密度差异一致")

In [ ]:
# ========== 可视化：两个城市通勤时间对比 ==========
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(common_labels))
width = 0.35

bars1 = ax.bar(x_pos - width/2, comparison['New York (%)'].values, width, 
               label='New York', color='#e74c3c', alpha=0.7)
bars2 = ax.bar(x_pos + width/2, comparison['Boise (%)'].values, width, 
               label='Boise', color='#2ecc71', alpha=0.7)

ax.set_xticks(x_pos)
ax.set_xticklabels(common_labels)
ax.set_xlabel('Commute Time (min)', fontsize=12)
ax.set_ylabel('Relative Frequency (%)', fontsize=12)
ax.set_title('Commute Time Distribution: New York vs Boise', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  红色（纽约）：通勤时间分布更均匀，长尾更明显")
print(f"  绿色（博伊西）：集中在短时间，大城市效应不明显")

---

## 📌 核心概念回顾

### 📌 频数分布表 (Frequency Distribution Table)
- **定义**: 通过展示数据类别（或组）以及每个类别中数据值的数量（频数），来显示数据是如何在不同类别间划分的
- **结构**: 组、组下限、组上限、组中值、组距、频数
- **作用**: 整理和汇总大型数据集，查看数据分布

### 📌 相对频数 (Relative Frequency)
- **公式**: $\text{相对频数} = \frac{\text{组频数}}{\text{数据总数}}$
- **含义**: 每组数据占总体的比例
- **应用**: 比较不同规模的数据集

### 📌 累积频数 (Cumulative Frequency)
- **定义**: 该组频数与前面所有组的频数之和
- **含义**: 表示"小于某值"的数据个数
- **应用**: 确定百分位数、中位数的位置

### 📌 正态分布的频数特征
- 频数开始时较低，接着上升到一个或两个高频数，最后下降到一个低频数
- 分布是近似对称的

### 🔗 完整流程
```
选择组数 → 计算组距 → 确定组界 → 统计频数 → 计算相对频数 → 计算累积频数
    ↓           ↓           ↓           ↓           ↓             ↓
  5~20组    (最大-最小)/组数  方便的数字  每组计数    频数/总数     累加求和
```

### 📝 检验指标汇总

| 指标 | 公式 | 用途 |
|------|------|------|
| 频数 | 计数 | 每组数据个数 |
| 相对频数 | 频数/n | 占总体比例 |
| 百分比 | 相对频数×100 | 百分比表示 |
| 累积频数 | 累加 | "小于某值"的个数 |
| 组中值 | (下限+上限)/2 | 代表每组的中心值 |

---

## ❌ 常见误区

### ❌ 误区 1: 组距 = 组上限 - 组下限
**✓ 正确理解**: 组距 = 下一组的组下限 - 本组的组下限。例如表 2-2 中，组距是 15（15-0=15），而不是 14（14-0=14）。

### ❌ 误区 2: 频数分布表会丢失原始数据信息
**✓ 正确理解**: 频数分布表确实会丢失一些细节，但它能揭示数据的整体分布形态，这在数据量大时比看原始数据更有价值。

### ❌ 误区 3: 组数越多越好
**✓ 正确理解**: 组数太多会导致每组频数太少，无法看出分布规律；组数太少会掩盖数据的细节特征。通常 5~20 组比较合适。

### ❌ 误区 4: 分类数据不能用频数分布表
**✓ 正确理解**: 频数分布表既适用于定量数据，也适用于分类数据。分类数据的频数分布表不需要组距和组中值。

### ❌ 误区 5: 相对频数之和不一定等于 1
**✓ 正确理解**: 相对频数之和必须等于 1（或百分比之和等于 100%），四舍五入导致的微小误差是可以接受的。